# Random-Label Control Table

Compact audit table for structured-random attentive controls. Missing models are shown as `NA`.

In [ ]:
from __future__ import annotations

import csv
import json
from pathlib import Path
from typing import Optional

try:
    import pandas as pd
except ModuleNotFoundError:
    pd = None


def find_repo_root(start: Optional[Path] = None) -> Path:
    current = Path.cwd() if start is None else start.resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'run.py').exists() and (candidate / 'configs').exists():
            return candidate
    raise FileNotFoundError('Could not find Probe4Physics repo root.')


REPO_ROOT = find_repo_root()
ARTIFACT_ROOTS = [
    REPO_ROOT / 'artifacts' / 'probes',
    Path('/scratch-shared/spunzo1/probe4physics/artifacts/probes'),
    Path('/gpfs/scratch1/shared/spunzo1/probe4physics/artifacts/probes'),
]

DATASETS = [
    {'key': 'intphys2', 'label': 'IntPhys2', 'metric': 'voe_accuracy', 'epochs': 90, 'patience': 20},
    {'key': 'mvp', 'label': 'MVP', 'metric': 'pair_consistency', 'epochs': 30, 'patience': 5},
]

MODELS = [
    {'label': 'V-JEPA', 'slug': 'jepa_v1_vith16_384'},
    {'label': 'V-JEPA 2', 'slug': 'jepa_v2_vitg_384'},
    {'label': 'V-JEPA 2.1', 'slug': 'jepa_v2_1_vitG_384'},
    {'label': 'VideoMAE', 'slug': 'videomae_vit_huge_16_224'},
    {'label': 'VideoMAE-v2', 'slug': 'videomae_v2_vit_giant_16_224'},
    {
        'label': 'LTX-Video',
        'slug_by_dataset': {
            'intphys2': 'ltx_video_ltxv_13b_0_9_8_distilled',
            'mvp': 'ltx_video_ltxv_13b_0_9_8_distilled',
        },
    },
]

In [ ]:
def model_slug(model: dict, dataset_key: str) -> str:
    return model.get('slug_by_dataset', {}).get(dataset_key, model.get('slug', ''))


def expected_group_name(dataset: dict, model: dict) -> str:
    slug = model_slug(model, dataset['key'])
    return (
        f"{dataset['key']}_probe_temporal_attn_{slug}_"
        f"structured_random_lr_matrix_ep{dataset['epochs']}_pat{dataset['patience']}"
    )


def candidate_group_dirs(dataset: dict, model: dict) -> list[Path]:
    group = expected_group_name(dataset, model)
    return [root / dataset['key'] / group for root in ARTIFACT_ROOTS]


def first_existing(paths: list[Path]) -> Optional[Path]:
    for path in paths:
        if path.exists():
            return path
    return None


def read_csv_rows(path: Path) -> list[dict[str, str]]:
    with path.open(newline='', encoding='utf-8') as handle:
        return list(csv.DictReader(handle))


def read_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding='utf-8'))


def as_float(value: object) -> Optional[float]:
    try:
        if value in (None, ''):
            return None
        return float(value)
    except (TypeError, ValueError):
        return None


def fmt(value: object) -> str:
    number = as_float(value)
    if number is None:
        return 'NA'
    return f'{number:.2f}'


def load_control_row(dataset: dict, model: dict) -> dict[str, str]:
    group_dir = first_existing(candidate_group_dirs(dataset, model))
    base = {
        'Dataset': dataset['label'],
        'Model': model['label'],
        'Best layer': 'NA',
        'LR': 'NA',
        'Train VOE/PC': 'NA',
        'Train acc': 'NA',
        'Val VOE/PC': 'NA',
        'Val acc': 'NA',
        'Test VOE/PC': 'NA',
        'Test acc': 'NA',
        'Status': 'NA',
    }
    if group_dir is None:
        return base

    metric = dataset['metric']
    summary_json = group_dir / 'train_eval_summary.json'
    if summary_json.exists():
        summary = read_json(summary_json)
        layers = [layer for layer in summary.get('layers', []) if isinstance(layer, dict)]
        if not layers:
            return {**base, 'Status': 'empty summary'}

        def split_metric(layer: dict, split: str, name: str) -> Optional[float]:
            metrics = layer.get('eval', {}).get('metrics_by_split', {}).get(split, {})
            if not isinstance(metrics, dict):
                return None
            return as_float(metrics.get(name))

        best = max(layers, key=lambda layer: split_metric(layer, 'val', metric) or float('-inf'))
        return {
            **base,
            'Best layer': str(best.get('layer_label') or best.get('layer') or 'NA'),
            'LR': str(best.get('learning_rate_tag') or best.get('learning_rate') or 'NA'),
            'Train VOE/PC': fmt(split_metric(best, 'train', metric)),
            'Train acc': fmt(split_metric(best, 'train', 'accuracy')),
            'Val VOE/PC': fmt(split_metric(best, 'val', metric)),
            'Val acc': fmt(split_metric(best, 'val', 'accuracy')),
            'Test VOE/PC': fmt(split_metric(best, 'test', metric)),
            'Test acc': fmt(split_metric(best, 'test', 'accuracy')),
            'Status': 'done',
        }

    summary_csv = group_dir / 'train_eval_summary.csv'
    if not summary_csv.exists():
        return {**base, 'Status': 'running/no summary'}

    rows = read_csv_rows(summary_csv)
    if not rows:
        return {**base, 'Status': 'empty summary'}

    best = max(rows, key=lambda row: as_float(row.get('selection_metric') or row.get(f'val_{metric}')) or float('-inf'))
    return {
        **base,
        'Best layer': best.get('layer_label') or best.get('layer') or 'NA',
        'LR': best.get('selected_lr_tag') or best.get('selected_lr') or 'NA',
        'Train VOE/PC': fmt(best.get(f'train_{metric}')),
        'Train acc': fmt(best.get('train_accuracy')),
        'Val VOE/PC': fmt(best.get(f'val_{metric}') or best.get('selection_metric')),
        'Val acc': fmt(best.get('val_accuracy')),
        'Test VOE/PC': fmt(best.get(f'test_{metric}') or best.get('objective_metric')),
        'Test acc': fmt(best.get('test_accuracy')),
        'Status': 'done',
    }


def make_table() -> list[dict[str, str]]:
    return [load_control_row(dataset, model) for dataset in DATASETS for model in MODELS]

In [ ]:
table_rows = make_table()

if pd is not None:
    table = pd.DataFrame(table_rows)
    display(table)
else:
    columns = list(table_rows[0])
    print('| ' + ' | '.join(columns) + ' |')
    print('| ' + ' | '.join(['---'] * len(columns)) + ' |')
    for row in table_rows:
        print('| ' + ' | '.join(str(row[col]) for col in columns) + ' |')

| Dataset | Model | Best layer | LR | Train VOE/PC | Train acc | Val VOE/PC | Val acc | Test VOE/PC | Test acc | Status |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| IntPhys2 | V-JEPA | 8 | 5e-4 | 18.54 | 53.31 | 29.41 | 57.84 | 9.80 | 50.49 | done |
| IntPhys2 | V-JEPA 2 | 20 | 5e-5 | 23.84 | 50.17 | 29.41 | 51.96 | 11.76 | 50.49 | done |
| IntPhys2 | V-JEPA 2.1 | 12 | 5e-4 | 19.21 | 50.00 | 27.45 | 53.43 | 9.80 | 47.06 | done |
| IntPhys2 | VideoMAE | 32 | 1e-5 | 34.44 | 51.66 | 29.41 | 54.41 | 13.73 | 50.49 | done |
| IntPhys2 | VideoMAE-v2 | 30 | 5e-4 | 16.56 | 50.00 | 25.49 | 53.92 | 19.61 | 52.94 | done |
| IntPhys2 | LTX-Video | 38 | 5e-5 | 23.18 | 50.50 | 29.41 | 52.94 | 17.65 | 50.49 | done |
| MVP | V-JEPA | 8 | 5e-4 | 12.28 | 49.34 | 14.05 | 50.61 | 13.85 | 51.06 | done |
| MVP | V-JEPA 2 | 30 | 5e-4 | 15.68 | 50.24 | 18.00 | 51.82 | 16.48 | 51.77 | done |
| MVP | V-JEPA 2.1 | 38 | 1e-4 | 18.28 | 50.99 | 18.81 | 50.96 | 16.99 | 49.04 | done |
| MVP 

In [ ]:
out_path = REPO_ROOT / 'results' / 'random_label_controls_table.csv'
out_path.parent.mkdir(parents=True, exist_ok=True)
with out_path.open('w', newline='', encoding='utf-8') as handle:
    writer = csv.DictWriter(handle, fieldnames=list(table_rows[0]))
    writer.writeheader()
    writer.writerows(table_rows)
out_path

PosixPath('/gpfs/home2/spunzo1/Probe4Physics/results/random_label_controls_table.csv')